# Compresión de Transformers para Análisis de Sentimiento en Dispositivos con Recursos Limitados

**Asignatura:** Investigación Aplicada en Deep Learning
**Temática:** Optimización de Transformers para Recursos Limitados
**Repositorio:** `AracelyKat/compresion-transformers-imdb`

---

### Objetivo
Comparar el efecto de la **distilación** y la **cuantización** sobre la precisión y la eficiencia
(tamaño, latencia y energía) de modelos Transformer en la tarea de clasificar reseñas de películas
como **positivas** o **negativas** (dataset IMDB).

### Pregunta de investigación
> ¿Cuánto se puede reducir el tamaño, el tiempo de inferencia y el consumo energético de BERT
> mediante distilación y cuantización, y cuál es el costo en precisión?

### Modelos que se comparan
| # | Modelo | Técnica |
|---|---|---|
| 1 | BERT-base FP32 | Referencia (sin comprimir) |
| 2 | BERT-base INT8 | Cuantización dinámica |
| 3 | DistilBERT FP32 | Distilación |
| 4 | DistilBERT INT8 | Distilación + cuantización |

### Contenido del notebook
0. Preparación del entorno *(Fase 0)*
1. Datos y preprocesamiento *(Fase 1)*
2. Entrenamiento de modelos base *(Fase 2)*
3. Compresión *(Fase 3)*
4. Evaluación experimental *(Fase 4)*
5. Visualización y análisis *(Fase 5)*
6. Conclusiones

> **Antes de ejecutar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.

---
# 0. Preparación del entorno

En esta sección se dejan listas las herramientas: librerías, conexión con Google Drive,
configuración general, semillas y registro del hardware. Todo lo necesario para que los
resultados se puedan **reproducir**.

## 0.1 Instalación de librerías

Colab ya trae PyTorch, `transformers`, scikit-learn y matplotlib. Instalamos o actualizamos lo que falta:

| Librería | Para qué se usa |
|---|---|
| `transformers` | Descargar y entrenar BERT y DistilBERT |
| `datasets` | Descargar y procesar el dataset IMDB |
| `evaluate` | Métricas durante el entrenamiento |
| `accelerate` | Requerida por el `Trainer` de Hugging Face |
| `codecarbon` | Estimar el consumo de energía (Fase 4) |
| `statsmodels` | Prueba estadística de McNemar (Fase 4) |

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate codecarbon statsmodels

## 0.2 Conexión con Google Drive

Colab **borra todos los archivos al cerrar la sesión**. Por eso guardamos en Google Drive los
modelos entrenados, los datos procesados y los resultados.

Se creará esta estructura en tu Drive (igual a la del repositorio):
```
MyDrive/deep-learning/
├── data/               ← datasets procesados
├── models/             ← modelos entrenados
└── results/
    ├── figures/        ← gráficos
    └── tables/         ← tablas (CSV / JSON)
```
Si el notebook se ejecuta fuera de Colab, se usan las carpetas locales del repositorio.

In [ ]:
from pathlib import Path

# Detectamos si estamos en Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Carpeta base: Drive en Colab, o la raíz del repositorio en local
BASE_DIR = Path('/content/drive/MyDrive/deep-learning') if IN_COLAB else Path('..').resolve()

DATA_DIR   = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
FIG_DIR    = BASE_DIR / 'results' / 'figures'
TAB_DIR    = BASE_DIR / 'results' / 'tables'

for carpeta in (DATA_DIR, MODELS_DIR, FIG_DIR, TAB_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)

print(f'¿En Colab?: {IN_COLAB}')
print(f'Carpeta base: {BASE_DIR}')

## 0.3 Configuración general

Reunimos en un solo lugar todos los valores que se usan en el notebook. Si hay que cambiar
algo (por ejemplo, el largo máximo de las reseñas), se cambia **solo aquí**.

In [ ]:
SEED = 42                 # Semilla para que los resultados se puedan repetir
MAX_LENGTH = 256          # Largo máximo de cada reseña en tokens (se justifica en la sección 1.6)
VAL_SIZE = 2_500          # Reseñas que se separan del entrenamiento para validar

# Modelos preentrenados de Hugging Face
MODELOS = {
    'bert':       'bert-base-uncased',
    'distilbert': 'distilbert-base-uncased',
}

ETIQUETAS = {0: 'negativa', 1: 'positiva'}

# Estilo de los gráficos: una paleta fija para todo el proyecto
COLOR_CLASE = {'negativa': '#eb6834', 'positiva': '#2a78d6'}  # naranja / azul (aptos para daltonismo)
COLOR_PRINCIPAL = '#2a78d6'
TINTA_SECUNDARIA = '#52514e'
TINTA_TENUE = '#898781'

## 0.4 Semillas y estilo de gráficos

Los modelos de Deep Learning usan números aleatorios (por ejemplo, al inicializar pesos o
mezclar datos). Fijar una **semilla** hace que cada ejecución produzca los mismos resultados.

In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import set_seed


def fijar_semilla(seed: int) -> None:
    """Fija la semilla de todas las librerías que usan números aleatorios."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)  # semilla interna de Hugging Face


fijar_semilla(SEED)

# Estilo común para todos los gráficos: limpio, con la cuadrícula en segundo plano
plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.titlelocation': 'left',
    'axes.edgecolor': TINTA_TENUE,
    'axes.labelcolor': TINTA_SECUNDARIA,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.axisbelow': True,
    'grid.color': '#e6e5e1',
    'grid.linewidth': 0.8,
    'xtick.color': TINTA_SECUNDARIA,
    'ytick.color': TINTA_SECUNDARIA,
    'legend.frameon': False,
})
print(f'Semilla fijada: {SEED}')

## 0.5 Registro del entorno (hardware y versiones)

Los tiempos de respuesta y la energía **dependen del hardware**, así que lo registramos y
guardamos en `results/tables/entorno.json`. Estos datos se citan en la sección de
Metodología del artículo.

In [ ]:
import json
import os
import platform
import sys

import datasets
import psutil
import sklearn
import transformers


def nombre_cpu() -> str:
    """Devuelve el modelo del procesador (en Linux/Colab se lee de /proc/cpuinfo)."""
    try:
        with open('/proc/cpuinfo') as f:
            for linea in f:
                if 'model name' in linea:
                    return linea.split(':', 1)[1].strip()
    except FileNotFoundError:
        pass
    return platform.processor() or 'desconocido'


entorno = {
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'datasets': datasets.__version__,
    'scikit-learn': sklearn.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU',
    'cpu': nombre_cpu(),
    'nucleos_cpu': os.cpu_count(),
    'hilos_torch': torch.get_num_threads(),
    'ram_gb': round(psutil.virtual_memory().total / 1e9, 1),
}

for clave, valor in entorno.items():
    print(f'{clave:>13}: {valor}')

with open(TAB_DIR / 'entorno.json', 'w') as f:
    json.dump(entorno, f, indent=2)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    print('\n⚠️ No hay GPU activa: el entrenamiento (Fase 2) será muy lento.')

---
# 1. Datos y preprocesamiento

**IMDB Movie Reviews** (Maas et al., 2011) contiene 50.000 reseñas de películas en inglés,
etiquetadas como positivas o negativas. En esta sección:

1. Cargamos el dataset.
2. Separamos un conjunto de validación.
3. Limpiamos el texto.
4. Exploramos los datos (EDA).
5. Elegimos el largo máximo de las reseñas.
6. Convertimos el texto en números (tokenización).

## 1.1 Carga del dataset

La librería `datasets` lo descarga directamente desde Hugging Face. Trae tres partes:
- `train`: 25.000 reseñas etiquetadas.
- `test`: 25.000 reseñas etiquetadas.
- `unsupervised`: 50.000 reseñas **sin etiqueta**, que **no usaremos**.

In [ ]:
from datasets import load_dataset

imdb = load_dataset('imdb')
imdb

## 1.2 División en entrenamiento, validación y prueba

| Conjunto | Para qué se usa | Tamaño |
|---|---|---|
| **Entrenamiento** | El modelo aprende de estos ejemplos | 22.500 |
| **Validación** | Revisamos el avance durante el entrenamiento y elegimos el mejor modelo | 2.500 |
| **Prueba** | Evaluación final. **No se toca hasta la Fase 4** | 25.000 |

La división es **estratificada**: se mantiene la misma proporción de positivas y negativas en
cada conjunto.

In [ ]:
from datasets import DatasetDict

division = imdb['train'].train_test_split(
    test_size=VAL_SIZE,
    seed=SEED,
    stratify_by_column='label',
)

datos = DatasetDict({
    'train': division['train'],
    'validation': division['test'],
    'test': imdb['test'],
})

for nombre, conjunto in datos.items():
    print(f'{nombre:>10}: {len(conjunto):>6,} reseñas')

## 1.3 Limpieza del texto

Las reseñas de IMDB traen etiquetas HTML como `<br />` (saltos de línea de la página web) que
no aportan significado. Hacemos una **limpieza mínima**:

- Quitar etiquetas HTML.
- Convertir entidades HTML (`&amp;` → `&`).
- Unificar los espacios en blanco.

**No** quitamos *stopwords* ("the", "not"…) ni signos de puntuación, porque BERT fue entrenado
con texto natural y palabras como **"not"** cambian por completo el sentimiento.

In [ ]:
import html
import re

PATRON_HTML = re.compile(r'<[^>]+>')
PATRON_ESPACIOS = re.compile(r'\s+')


def limpiar_texto(texto: str) -> str:
    """Elimina etiquetas HTML, decodifica entidades y unifica espacios."""
    texto = PATRON_HTML.sub(' ', texto)
    texto = html.unescape(texto)
    return PATRON_ESPACIOS.sub(' ', texto).strip()


# Ejemplo: buscamos una reseña que tenga etiquetas HTML
ejemplo = next(t for t in datos['train']['text'] if '<br' in t)
print('ANTES:\n', ejemplo[:400], '\n')
print('DESPUÉS:\n', limpiar_texto(ejemplo)[:400])

In [ ]:
# Aplicamos la limpieza a los tres conjuntos (en lotes, para que sea más rápido)
datos = datos.map(
    lambda lote: {'text': [limpiar_texto(t) for t in lote['text']]},
    batched=True,
    desc='Limpiando',
)

quedan_html = sum('<br' in t for t in datos['train']['text'])
print(f'Reseñas de entrenamiento que aún tienen <br>: {quedan_html}')

## 1.4 Análisis exploratorio: balance de clases

Comprobamos si hay la misma cantidad de reseñas positivas y negativas. Si las clases estuvieran
desbalanceadas, la *accuracy* podría ser engañosa y habría que darle más peso al F1-score.

In [ ]:
import pandas as pd

conteo = pd.DataFrame({
    nombre: pd.Series(conjunto['label']).map(ETIQUETAS).value_counts()
    for nombre, conjunto in datos.items()
}).T[['negativa', 'positiva']]

conteo['% positivas'] = (conteo['positiva'] / conteo.sum(axis=1) * 100).round(1)
conteo

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))

nombres = {'train': 'Entrenamiento', 'validation': 'Validación', 'test': 'Prueba'}
x = np.arange(len(conteo))
ancho = 0.38

for i, clase in enumerate(['negativa', 'positiva']):
    barras = ax.bar(
        x + (i - 0.5) * ancho, conteo[clase], width=ancho - 0.02,
        color=COLOR_CLASE[clase], label=clase.capitalize(),
    )
    ax.bar_label(barras, labels=[f'{v:,}' for v in conteo[clase]],
                 padding=2, fontsize=8, color=TINTA_SECUNDARIA)

ax.set_xticks(x, [nombres[n] for n in conteo.index])
ax.set_ylabel('Número de reseñas')
ax.set_title('Distribución de clases por conjunto')
ax.grid(axis='x', visible=False)
ax.legend(loc='upper left', ncols=2)
ax.set_ylim(0, conteo[['negativa', 'positiva']].values.max() * 1.15)

fig.savefig(FIG_DIR / 'eda_clases.png')
plt.show()

## 1.5 Análisis exploratorio: longitud de las reseñas

BERT y DistilBERT **no pueden leer textos de cualquier largo**: su máximo es 512 *tokens*
(un token es una palabra o un trozo de palabra). Además, cuanto más largo el texto, más
memoria y tiempo consumen. Por eso medimos la longitud de las reseñas:

- En **palabras**, para tener una idea intuitiva.
- En **tokens**, que es lo que realmente procesa el modelo.

Usamos solo el conjunto de entrenamiento, para no "espiar" los datos de prueba.

In [ ]:
from transformers import AutoTokenizer

tokenizer_bert = AutoTokenizer.from_pretrained(MODELOS['bert'])

# Ejemplo de cómo BERT divide una frase en tokens
frase = 'This movie was absolutely unwatchable!'
print('Frase :', frase)
print('Tokens:', tokenizer_bert.tokenize(frase))

In [ ]:
textos_train = datos['train']['text']
etiquetas_train = np.array(datos['train']['label'])

# Largo en palabras y en tokens de cada reseña
# (+2 tokens especiales que BERT agrega: [CLS] al inicio y [SEP] al final)
long_palabras = np.array([len(t.split()) for t in textos_train])
long_tokens = np.array([
    len(ids) for ids in tokenizer_bert(textos_train, add_special_tokens=True, verbose=False)['input_ids']
])

resumen_longitud = pd.DataFrame({
    'palabras': pd.Series(long_palabras).describe(percentiles=[.5, .75, .9, .95]),
    'tokens': pd.Series(long_tokens).describe(percentiles=[.5, .75, .9, .95]),
}).round(0)
resumen_longitud

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
LIMITES = [128, 256, 512]

# (a) Histograma: ¿cuántas reseñas tienen cada largo?
ax1.hist(long_tokens, bins=np.arange(0, 1001, 20), color=COLOR_PRINCIPAL,
         edgecolor='white', linewidth=0.6)
for lim in LIMITES:
    ax1.axvline(lim, color=TINTA_SECUNDARIA, linestyle='--', linewidth=1)
    ax1.text(lim + 8, ax1.get_ylim()[1] * 0.92, f'{lim}', color=TINTA_SECUNDARIA, fontsize=8)
ax1.set_xlim(0, 1000)
ax1.set_xlabel('Largo de la reseña (tokens)')
ax1.set_ylabel('Número de reseñas')
ax1.set_title('(a) Distribución del largo')
ax1.grid(axis='x', visible=False)
ax1.text(0.98, 0.55, f'{(long_tokens > 1000).sum()} reseñas\n> 1000 tokens\nno se muestran',
         transform=ax1.transAxes, ha='right', fontsize=8, color=TINTA_TENUE)

# (b) Curva acumulada por clase: ¿qué % de reseñas cabe completa en N tokens?
for valor, clase in ETIQUETAS.items():
    largos = np.sort(long_tokens[etiquetas_train == valor])
    acumulado = np.arange(1, len(largos) + 1) / len(largos) * 100
    ax2.plot(largos, acumulado, color=COLOR_CLASE[clase], linewidth=2, label=clase.capitalize())
for lim in LIMITES:
    ax2.axvline(lim, color=TINTA_SECUNDARIA, linestyle='--', linewidth=1)
ax2.set_xlim(0, 1000)
ax2.set_ylim(0, 101)
ax2.set_xlabel('Largo máximo permitido (tokens)')
ax2.set_ylabel('% de reseñas que caben completas')
ax2.set_title('(b) Cobertura acumulada por clase')
ax2.legend(loc='lower right')

fig.tight_layout()
fig.savefig(FIG_DIR / 'eda_longitud.png')
plt.show()

print(f'Largo promedio (tokens) → negativas: {long_tokens[etiquetas_train == 0].mean():.0f} | '
      f'positivas: {long_tokens[etiquetas_train == 1].mean():.0f}')

## 1.6 Elección del largo máximo (`MAX_LENGTH`)

Las reseñas más largas que el límite se **recortan** (truncamiento): se conserva el inicio y se
pierde el final. Comparamos tres límites posibles:

In [ ]:
cobertura = pd.DataFrame({
    'max_length': LIMITES,
    '% reseñas completas': [(long_tokens <= lim).mean() * 100 for lim in LIMITES],
    '% reseñas recortadas': [(long_tokens > lim).mean() * 100 for lim in LIMITES],
}).round(1)

cobertura.to_csv(TAB_DIR / 'eda_cobertura_tokens.csv', index=False)
cobertura

**Decisión: `MAX_LENGTH = 256`.**

- Con **256** tokens, la mayoría de las reseñas cabe completa (ver tabla), y las recortadas
  conservan suficiente texto para captar el sentimiento.
- Con **512** casi todas caben, pero el costo de la autoatención crece con el **cuadrado** de la
  longitud: duplicar el largo implica aproximadamente **4 veces** más cálculo en esas capas. Eso
  va en contra del objetivo del proyecto (modelos eficientes).
- Con **128** se recorta demasiado texto.

> 📝 **Para el artículo (limitación):** al recortar se pierde el final de las reseñas largas,
> donde a veces está la conclusión del autor ("...en resumen, no la recomiendo"). Como trabajo
> futuro se podría conservar el inicio **y** el final (Sun et al., 2019).

## 1.7 Ejemplos de reseñas

Leemos algunas reseñas para entender el tipo de lenguaje: opiniones informales, sarcasmo,
mezcla de elogios y críticas, etc.

In [ ]:
import textwrap

muestra = datos['train'].shuffle(seed=SEED)
for valor, clase in ETIQUETAS.items():
    ejemplos = muestra.filter(lambda e: e['label'] == valor).select(range(2))
    for i, ej in enumerate(ejemplos, 1):
        print(f'━━━ Reseña {clase.upper()} #{i} ━━━')
        print(textwrap.fill(ej['text'][:500] + ('…' if len(ej['text']) > 500 else ''), width=100))
        print()

## 1.8 Tokenización

Los modelos no leen palabras, sino **números**. El *tokenizador* de cada modelo convierte cada
reseña en una lista de identificadores (`input_ids`) y una máscara de atención
(`attention_mask`) que indica qué posiciones son texto real y cuáles son relleno.

- **Truncamiento:** las reseñas de más de `MAX_LENGTH` tokens se recortan.
- **Relleno (*padding*) dinámico:** no se rellena aquí, sino al armar cada lote durante el
  entrenamiento, solo hasta la reseña más larga **de ese lote**. Así se ahorra memoria y tiempo
  frente a rellenar todas las reseñas hasta 256.

> 💡 BERT-base *uncased* y DistilBERT *uncased* usan **el mismo vocabulario**, así que generan
> los mismos `input_ids`. La diferencia es que BERT también usa `token_type_ids`. Igual
> tokenizamos por separado con el tokenizador oficial de cada modelo, por buena práctica.

In [ ]:
def tokenizar(conjunto: DatasetDict, nombre_modelo: str) -> DatasetDict:
    """Tokeniza los tres conjuntos con el tokenizador del modelo indicado."""
    tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)
    return conjunto.map(
        lambda lote: tokenizer(lote['text'], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        desc=f'Tokenizando ({nombre_modelo})',
    )


datos_tok = {clave: tokenizar(datos, nombre) for clave, nombre in MODELOS.items()}

# Verificación: ninguna reseña supera MAX_LENGTH
for clave, dt in datos_tok.items():
    largo_max = max(len(ids) for ids in dt['train']['input_ids'])
    print(f'{clave:>10} → columnas: {dt["train"].column_names} | largo máximo: {largo_max}')

In [ ]:
# Ejemplo: una reseña convertida en números
ej = datos_tok['bert']['train'][0]
print('Texto      :', datos['train'][0]['text'][:120], '…')
print('input_ids  :', ej['input_ids'][:15], '…')
print('tokens     :', tokenizer_bert.convert_ids_to_tokens(ej['input_ids'][:15]), '…')
print('Nº tokens  :', len(ej['input_ids']))

## 1.9 Guardado de los datos procesados

Guardamos en Drive el dataset limpio y las versiones tokenizadas. En las siguientes fases se
cargan directamente, sin repetir el preprocesamiento.

In [ ]:
datos.save_to_disk(DATA_DIR / 'imdb_limpio')
for clave, dt in datos_tok.items():
    dt.save_to_disk(DATA_DIR / f'imdb_tok_{clave}')

print('Guardado en:', DATA_DIR)
for ruta in sorted(DATA_DIR.iterdir()):
    print('  -', ruta.name)

## 1.10 Conclusiones de la Fase 1

> ✏️ **Completar después de ejecutar**, con los números obtenidos:

- **Balance de clases:** …
- **Longitud de las reseñas:** mediana de … tokens; el …% supera los 256 tokens.
- **¿Las positivas y negativas tienen largos distintos?** …
- **Decisión:** `MAX_LENGTH = 256`, porque …
- **Datos listos:** 22.500 de entrenamiento, 2.500 de validación y 25.000 de prueba, tokenizados para ambos modelos.

---
# 2. Entrenamiento de modelos base

*Pendiente: se desarrolla en la Fase 2.*

---
# 3. Compresión

*Pendiente: se desarrolla en la Fase 3.*

---
# 4. Evaluación experimental

*Pendiente: se desarrolla en la Fase 4.*

---
# 5. Visualización y análisis

*Pendiente: se desarrolla en la Fase 5.*

---
# 6. Conclusiones

*Pendiente: se desarrolla en la Fase 5.*